# FULL text — shared preprocessing base (full_v1)

Produce FULL text for all 114 selected cases, including pending Sodhi, without changing
case IDs, source files, labels, or the frozen split manifest. FULL preserves the
controlling opinion's facts, reasoning, citations, procedural history, disposition,
majority footnotes, and signed-author attribution.

The cleaner removes recognized publication/caption/counsel metadata, page markers,
PDF running headers, judicial rosters, clerk certification, publisher prior-history
brackets, and known web markup. Only an explicit, unambiguous separate dissent
heading permits removal of dissent text. References to dissent within majority
reasoning are retained. Ambiguous boundaries are preserved and flagged.

Rules were developed using the 81 frozen training cases and synthetic structural
checks, then frozen in `preprocessing/full_rules_v1.json` before applying them to
all cases. No gold labels or annotation evidence select removals. The same cleaner
and version apply to both classes and every split; no evaluation-specific overrides.

**All later conditions must use these exact FULL files as their common base.**
MASKED will apply additional removal to FULL, not a separately cleaned source.
No MASKED files or model inputs containing annotation features are generated here.

Run all cells with Python 3 from MVP or any directory inside the repository. Keep
`preprocessing/` beside the notebook. Only the standard library is required.


In [1]:
from pathlib import Path
import importlib.util
import hashlib
import json
import sys
sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None: raise FileNotFoundError("Run inside legal-judgment.")
MVP = ROOT / "annotation/submissions/becca/MVP"
spec = importlib.util.spec_from_file_location("mvp_full_cleaner", MVP / "preprocessing/full_text.py")
cleaner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cleaner)
policy = json.loads((MVP / "preprocessing/full_rules_v1.json").read_text())
cleaner.require(cleaner.digest((MVP / "preprocessing/full_text.py").read_bytes()) == policy["cleaner_sha256"], "Frozen cleaner changed")
print("Rule version:", policy["version"])
print("Rule-development training cases:", len(policy["training_case_ids"]))


Rule version: full_cleaner_v1
Rule-development training cases: 81


## Generate FULL files and the removal audit

`full.jsonl` contains only `case_id` and `text`; individual files live in `texts/`.
Audit rows preserve original-source character offsets, the exact affected text,
removal reasons, replacements, before/after word counts, and hashes. Whitespace is
collapsed after applying recorded edits; no substantive text is rewritten.

Existing versioned files cannot be overwritten with changed results. Later rule or
source changes require a new version. Source OCR errors and split words are not
silently repaired; the audit retains their original context.


In [2]:
summary, audits = cleaner.build(ROOT)
print(json.dumps(summary, indent=2))
print("Net words removed:", summary["source_words"] - summary["full_words"])
print("Net word reduction: {:.2%}".format(1-summary["full_words"]/summary["source_words"]))
print("Minimum / maximum FULL words:", min(a["full_words"] for a in audits), max(a["full_words"] for a in audits))


{
  "case_count": 114,
  "rule_version": "full_cleaner_v1",
  "removal_reasons": {
    "publication_caption_counsel_header": 79,
    "publication_page_marker": 82,
    "judicial_concurrence_roster": 78,
    "judicial_panel_roster": 33,
    "pdf_running_page_docket": 3,
    "clerk_certification": 3,
    "publisher_prior_case_history": 3
  },
  "flag_counts": {
    "dissent_reference_in_retained_text_check_context": 2
  },
  "source_words": 72105,
  "full_words": 63619,
  "separate_dissent_cases": [],
  "label_fields_in_model_text_export": false,
  "frozen_ids_and_splits_changed": false,
  "masked_generated": false
}
Net words removed: 8486
Net word reduction: 11.77%
Minimum / maximum FULL words: 149 1796


## Validate retained order, coverage, and removal boundaries

For every case, reconstruct FULL by applying the audited source-offset edits in
order and normalizing whitespace. Compare this reconstruction with both exports;
verify source and output hashes and exact 114-ID coverage. Dispositions remain in
FULL; their removal belongs only to a later MASKED step.

The synthetic cases below check a separate dissent, an ambiguous boundary followed
by an order, HTML cleanup/entity decoding, a majority's reference to dissent, and a
judge roster followed by a controlling order and substantive footnote. They do not
use evaluation cases or gold labels to develop a new rule.


In [3]:
source_rows = cleaner.read_rows(MVP / "data/unanimous_include_v1/cases.jsonl")
sources = {r["case_id"]: r for r in source_rows}
full_rows = cleaner.read_rows(MVP / "data/full_v1/full.jsonl")
full = {r["case_id"]: r["text"] for r in full_rows}
cleaner.require(len(full_rows) == len(full) == 114 and set(full) == set(sources), "Export ID mismatch")
for a in audits:
    cid = a["case_id"]; source = sources[cid]["text"]
    cleaner.require(cleaner.apply_edits(source, a["edits"]) == full[cid], "Retained text/order mismatch")
    cleaner.require(cleaner.digest(full[cid].encode()) == a["full_sha256"], "FULL text hash mismatch")
    cleaner.require((MVP / "data/full_v1/texts" / (cid + ".txt")).read_text() == full[cid] + "\n", "Text export mismatch")
    cleaner.require(full[cid].strip() and a["full_words"] >= 50, "Empty/near-empty text")
manifest = json.loads((MVP / "data/full_v1/manifest.json").read_text())
for p, expected in manifest["inputs"].items():
    cleaner.require(cleaner.digest((ROOT / p).read_bytes()) == expected, "Input checksum mismatch")
for p, expected in manifest["outputs"].items():
    cleaner.require(cleaner.digest((MVP / "data/full_v1" / p).read_bytes()) == expected, "Output checksum mismatch")

majority = "Order, Supreme Court, affirmed. " + "The controlling opinion preserves these facts and this reasoning. " * 15
out, audit = cleaner.clean_full(majority + " Smith, J. (dissenting). I would reverse.")
cleaner.require(out == cleaner.normalize_space(majority), "Separate dissent test failed")
out, audit = cleaner.clean_full(majority + " Smith, J. (dissenting). I disagree. Ordered that the order is affirmed.")
cleaner.require("Ordered that" in out and "ambiguous_dissent_boundary_preserved" in audit["flags"], "Ambiguous order lost")
out, audit = cleaner.clean_full("<nav>Web menu</nav><p>" + majority + " The majority rejects the dissent. A &amp; B.</p>")
cleaner.require("Web menu" not in out and "<p>" not in out and "A & B" in out and "rejects the dissent" in out, "HTML/majority-reference test failed")
out, audit = cleaner.clean_full(majority + " Rose, J.P, Malone Jr., Stein and Egan Jr., JJ, concur. Ordered that the order is reversed. Footnote: substantive explanation.")
cleaner.require("Rose, J.P" not in out and "Ordered that" in out and "substantive explanation" in out, "Roster removal cut controlling text")
print("Passed: all 114 exports, source-order reconstruction, hashes, nonempty text, and structural regression cases.")


Passed: all 114 exports, source-order reconstruction, hashes, nonempty text, and structural regression cases.


## Review findings and limitations

- 79 publication/caption/counsel prefixes, 82 publication page markers, 111 judicial
  rosters, 3 PDF running headers, 3 clerk certifications, and 3 publisher prior-history
  brackets were removed: **281 recorded spans** in total.
- **No literal web tags or separate dissent sections required removal in these
  already-extracted source texts.** This does not establish that the complete
  original publications contain no dissent.
- `NY-2682947` and `NY-5997418` contain majority responses to dissent. Both source
  texts were read in full; the retained references are majority reasoning, not a
  separate dissent. Their diagnostic flags remain visible in the raw audit, with
  resolutions in `preprocessing/full_review_v1.json`.
- Signed majority attribution and substantive footnotes remain. The training
  McNally opinion (`NY-5985468`) specifically verifies that a roster can precede
  a controlling disposition and footnotes; those later passages are preserved.
- All 114 outputs are nonempty and at least 149 words. Original OCR quirks remain;
  this is structural cleaning, not a new transcription or human text adjudication.

Files: `data/full_v1/full.jsonl`, `texts/<case_id>.txt`, `removal_audit.jsonl`,
`summary.json`, and `manifest.json`. The prior `mvp_split_v1` manifest and its
pending/excluded statuses remain unchanged. Use only the same frozen assigned IDs
and labels across experiments; inclusion in this text export does not grant
permission to train on a pending case.
